# T4 — Updated Latency: Pressure Time Series

Interactive time series showing heel and toe pressure for each foot.
Use the dropdown(s) to switch between participants.

## Study phase context

**T4** addressed system latency between the pressure insole and the audio playback. Ten participants walked while the system measured the delay between a foot event and the corresponding audio cue. The measured latency was used to update the audio scheduling in subsequent trials.

Correcting this latency was essential for the tempo and weight manipulations in T5 and T6 to be synchronised with actual foot-strike events.

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
DATA_DIR = Path(".")

In [ ]:
def parse_pressure_file(path):
    """Parse any semicolon-delimited pressure CSV into time_s, L_toe, L_heel, R_toe, R_heel."""
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    raw = path.read_bytes()
    data = raw.replace(b"
", b"
").replace(b"", b"
").decode("utf-8", errors="replace")
    records = [r.strip() for r in data.split(";") if r.strip()]
    rows = []
    for rec in records:
        rec = re.sub(r"^\d+,\s*", "", rec)   # strip leading index field
        nums = re.findall(r"[\d]+\.?[\d]*", rec)
        if len(nums) < 5:
            continue
        try:
            ts = float(nums[0])
            sensors = [float(n) for n in nums[1:] if float(n) <= 1023]
            if len(sensors) < 4:
                continue
            rows.append({"ts_ms": ts, "L_toe": sensors[0], "L_heel": sensors[1],
                         "R_toe": sensors[2], "R_heel": sensors[3]})
        except ValueError:
            continue
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df["time_s"] = (df["ts_ms"] - df["ts_ms"].iloc[0]) / 1000.0
    return df


def parse_imu_pressure_file(path):
    """Parse IMU-derived pressure_from_imu.csv (device_ms,toe_L,heel_L,toe_R,heel_R)."""
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path)
    df = df.rename(columns={"toe_L": "L_toe", "heel_L": "L_heel",
                             "toe_R": "R_toe", "heel_R": "R_heel"})
    for col in ["L_toe", "L_heel", "R_toe", "R_heel"]:
        if col not in df.columns:
            df[col] = float("nan")
    df = df.dropna(subset=["device_ms"]).reset_index(drop=True)
    df["time_s"] = (df["device_ms"] - df["device_ms"].iloc[0]) / 1000.0
    return df

In [ ]:
def plot_pressure(df, title=""):
    fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
    fig.subplots_adjust(hspace=0.38)
    TOE_COLOR  = "#2ecc71"
    HEEL_COLOR = "#e74c3c"
    for ax, (foot, toe_col, heel_col) in zip(axes, [
        ("Left",  "L_toe", "L_heel"),
        ("Right", "R_toe", "R_heel"),
    ]):
        if df is None or df.empty or toe_col not in df.columns:
            ax.text(0.5, 0.5, "No data available", ha="center", va="center",
                    transform=ax.transAxes, fontsize=11, color="gray")
            ax.set_title(f"{foot} foot", fontsize=10)
            continue
        ax.plot(df["time_s"], df[toe_col],  color=TOE_COLOR,  lw=0.9,
                label=f"{foot} toe",  alpha=0.85)
        ax.plot(df["time_s"], df[heel_col], color=HEEL_COLOR, lw=0.9,
                label=f"{foot} heel", alpha=0.85)
        ax.set_ylabel("Pressure (ADC)", fontsize=9)
        ax.set_title(f"{foot} foot", fontsize=10)
        ax.legend(fontsize=8, loc="upper right")
        ax.grid(linestyle="--", alpha=0.35)
    axes[-1].set_xlabel("Time (s)", fontsize=10)
    fig.suptitle(title, fontsize=11, y=1.01)
    plt.show()

## Participant / data discovery

In [ ]:
EXCLUDED = {"excluded data"}
participants = sorted([
    d.name for d in DATA_DIR.iterdir()
    if d.is_dir() and d.name not in EXCLUDED
])

def phases_for(pid):
    d = DATA_DIR / pid
    available = []
    if list(d.glob("calibration-*.csv")): available.append("calibration")
    if list(d.glob("*-updated.csv")):     available.append("updated")
    return available or ["(none)"]

print(f"Participants: {participants}")
for p in participants:
    print(f"  {p}: {phases_for(p)}")

## Interactive time series

In [ ]:
def load_data_t4(pid, phase):
    d = DATA_DIR / pid
    if phase == "calibration":
        files = sorted(d.glob("calibration-*.csv"))
    elif phase == "updated":
        files = sorted(d.glob("*-updated.csv"))
    else:
        return pd.DataFrame()
    dfs = []
    offset_ms = 0.0
    for f in files:
        df = parse_pressure_file(f)
        if df.empty:
            continue
        df["ts_ms"] += offset_ms
        offset_ms = df["ts_ms"].iloc[-1] + 500.0
        dfs.append(df)
    if not dfs:
        return pd.DataFrame()
    combined = pd.concat(dfs, ignore_index=True)
    combined["time_s"] = (combined["ts_ms"] - combined["ts_ms"].iloc[0]) / 1000.0
    return combined

init_pid    = participants[0] if participants else ""
init_phases = phases_for(init_pid) if init_pid else ["(none)"]

pid_sel = widgets.Dropdown(
    options=participants, value=init_pid,
    description="Participant:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="200px"),
)
phase_sel = widgets.Dropdown(
    options=init_phases, value=init_phases[0],
    description="Phase:",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="180px"),
)
out = widgets.Output()

def refresh(_=None):
    phase_sel.options = phases_for(pid_sel.value)
    out.clear_output(wait=True)
    with out:
        df = load_data_t4(pid_sel.value, phase_sel.value)
        plot_pressure(df, title=f"T4 — P{pid_sel.value}  |  {phase_sel.value}")

pid_sel.observe(refresh, names="value")
phase_sel.observe(refresh, names="value")
display(widgets.HBox([pid_sel, phase_sel]), out)
refresh()